In [ ]:
import pandas as pd

# 1. Cargar el dataset limpio de la Parte 1
# Asegúrate de que el archivo 'telecom_churn_clean.csv' esté subido a tu entorno de Colab
archivo_csv = 'telecom_churn_clean.csv'

print(f"🔄 Intentando cargar los datos desde: {archivo_csv}...\n")

try:
    df_model = pd.read_csv(archivo_csv)
    print("✅ ¡Datos limpios cargados exitosamente!")

    # 2. Verificar dimensiones y estructura
    print(f"📊 Dimensiones del dataset preparado: {df_model.shape}\n")

    print("--- Vista previa de las primeras 5 filas ---")
    display(df_model.head())

    print("\n--- Verificación de Tipos de Datos (Info) ---")
    df_model.info()

except FileNotFoundError:
    print(f"❌ Error: No se encontró el archivo '{archivo_csv}'.")
    print("💡 Tip: Si reiniciaste Google Colab, debes volver a subir el archivo CSV usando el ícono de la carpeta en el menú de la izquierda.")

In [ ]:
import pandas as pd
import json

archivo = 'base de datos.txt'

print(f"🔄 Cargando y procesando '{archivo}'...")

try:
    # 1. Cargar el archivo JSON
    with open(archivo, 'r') as file:
        datos_crudos = json.load(file)

    # 2. Aplanar los datos (Convierte el JSON anidado en una tabla plana)
    df_ml = pd.json_normalize(datos_crudos)

    # 3. Limpiar los nombres de las columnas
    # json_normalize crea nombres como 'account.Charges.Total'.
    # Aquí nos quedamos solo con la última parte de la palabra (ej: 'Total')
    df_ml.columns = [col.split('.')[-1] for col in df_ml.columns]

    # 4. Re-aplicar la limpieza clave de la Parte 1
    # Convertir 'Total' a numérico y rellenar nulos con 0
    df_ml['Total'] = pd.to_numeric(df_ml['Total'], errors='coerce').fillna(0)

    # Crear la variable objetivo numérica (1 = Churn, 0 = No Churn)
    df_ml['Churn_Target'] = df_ml['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

    # Normalizar "No internet service" a "No" en servicios adicionales
    cols_servicios = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
    for col in cols_servicios:
        df_ml[col] = df_ml[col].replace('No internet service', 'No')

    print("✅ ¡Datos cargados, aplanados y limpios exitosamente!\n")

    # 5. Verificación de preparación
    print(f"📊 Dimensiones del dataset listo para ML: {df_ml.shape}\n")

    # Mostrar una muestra de las columnas más importantes
    columnas_clave = ['customerID', 'tenure', 'Monthly', 'Total', 'Churn', 'Churn_Target']
    display(df_ml[columnas_clave].head())

except FileNotFoundError:
    print(f"❌ Error: No se encontró el archivo '{archivo}'.")
    print("💡 Tip: Recuerda subir 'base de datos.txt' al entorno de Colab usando el menú de la izquierda (ícono de carpeta).")

In [ ]:
# 1. Definir las columnas que no aportan valor predictivo o son redundantes
columnas_a_eliminar = ['customerID', 'Churn']

print(f"🧹 Eliminando las siguientes columnas: {columnas_a_eliminar}...")

# 2. Eliminar las columnas usando el método .drop()
# axis=1 indica que estamos eliminando columnas (no filas)
df_ml = df_ml.drop(columns=columnas_a_eliminar, axis=1)

# 3. Verificación de las columnas restantes
print("\n✅ Columnas eliminadas con éxito.")
print(f"📊 Nuevas dimensiones del dataset: {df_ml.shape}\n")

print("--- Lista de variables finales para el modelo ---")
print(df_ml.columns.tolist())

In [ ]:
print("⚙️ Aplicando One-Hot Encoding a las variables categóricas...\n")

# 1. Aplicar One-Hot Encoding
# Pandas detectará automáticamente las columnas de texto (object)
# dtype=int asegura que los resultados sean 1 y 0 en lugar de True/False
df_ml_encoded = pd.get_dummies(df_ml, drop_first=True, dtype=int)

# 2. Verificación de las nuevas dimensiones
print("✅ Transformación completada con éxito.")
print(f"📊 Dimensiones anteriores: {df_ml.shape}")
print(f"📊 Nuevas dimensiones: {df_ml_encoded.shape}\n")

# 3. Ver cómo quedaron las columnas
print("--- Muestra de los datos codificados ---")
display(df_ml_encoded.head())

# 4. Ver los nombres de algunas de las nuevas columnas creadas
print("\n--- Nombres de las columnas generadas (Ejemplo) ---")
print(df_ml_encoded.columns.tolist()[:15]) # Mostramos las primeras 15

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("⚖️ Evaluando el balance de la variable objetivo (Churn_Target)...\n")

# 1. Calcular conteos absolutos
conteo_clases = df_ml_encoded['Churn_Target'].value_counts()
print("--- Conteo Absoluto de Clientes ---")
print(f"Se quedaron (0): {conteo_clases[0]} clientes")
print(f"Cancelaron  (1): {conteo_clases[1]} clientes")

# 2. Calcular proporciones (porcentajes) usando normalize=True
proporcion_clases = df_ml_encoded['Churn_Target'].value_counts(normalize=True) * 100
print("\n--- Proporción de Clases (%) ---")
print(f"Se quedaron (0): {proporcion_clases[0]:.2f}%")
print(f"Cancelaron  (1): {proporcion_clases[1]:.2f}%\n")

# 3. Visualización del desbalance
plt.figure(figsize=(7, 5))
# Usamos el dataframe codificado que venimos trabajando
ax = sns.countplot(x='Churn_Target', data=df_ml_encoded, palette=['#2ca02c', '#d62728'])

plt.title('Desbalance de Clases: Clientes Activos vs. Cancelados', fontsize=14)
plt.xlabel('Estado del Cliente (0 = Retenido | 1 = Canceló)', fontsize=12)
plt.ylabel('Cantidad de Clientes', fontsize=12)

# Agregar los porcentajes visualmente sobre las barras
total = len(df_ml_encoded)
for p in ax.patches:
    porcentaje = f'{100 * p.get_height() / total:.1f}%\n'
    x_pos = p.get_x() + p.get_width() / 2
    y_pos = p.get_height()
    ax.annotate(porcentaje, (x_pos, y_pos), ha='center', va='center', fontsize=12, fontweight='bold', color='black')

plt.ylim(0, max(conteo_clases) * 1.15) # Dar espacio arriba para el texto
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import pandas as pd

print("🚀 Iniciando proceso de división y balanceo con SMOTE...\n")

# 1. Separar las características (X) y la variable objetivo (y)
X = df_ml_encoded.drop('Churn_Target', axis=1)
y = df_ml_encoded['Churn_Target']

# 2. Dividir en Entrenamiento (80%) y Prueba (20%)
# stratify=y asegura que la proporción original de churn se mantenga en ambos sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("📊 Distribución en el set de ENTRENAMIENTO (Antes de SMOTE):")
print(y_train.value_counts())
print("\n")

# 3. Aplicar SMOTE SOLO al set de entrenamiento
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 4. Verificar el resultado del balanceo
print("⚖️ Distribución en el set de ENTRENAMIENTO (Después de SMOTE):")
print(y_train_smote.value_counts())

# 5. Verificación de las dimensiones finales
print("\n✅ Resumen de dimensiones finales listas para ML:")
print(f"X_train_smote: {X_train_smote.shape} (Entrenamiento balanceado)")
print(f"y_train_smote: {y_train_smote.shape}")
print(f"X_test:        {X_test.shape} (Prueba original - mundo real)")
print(f"y_test:        {y_test.shape}")

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

print("📏 Iniciando la estandarización de los datos...\n")

# 1. Inicializar el escalador
scaler = StandardScaler()

# 2. Entrenar (fit) el escalador SOLO con los datos de entrenamiento balanceados
# y transformarlos (transform) al mismo tiempo
X_train_scaled_array = scaler.fit_transform(X_train_smote)

# 3. Transformar el set de prueba usando las reglas aprendidas del set de entrenamiento
X_test_scaled_array = scaler.transform(X_test)

# 4. (Opcional pero recomendado) Convertir los arrays resultantes de vuelta a DataFrames
# para mantener los nombres de las columnas. Esto es vital para ver la "Importancia de Variables" después.
X_train_scaled = pd.DataFrame(X_train_scaled_array, columns=X_train_smote.columns)
X_test_scaled = pd.DataFrame(X_test_scaled_array, columns=X_test.columns)

print("✅ Datos estandarizados correctamente.")
print("\n--- Muestra de los datos escalados (X_train_scaled) ---")
# Notarás que los valores ahora están centrados alrededor de 0 (con valores negativos y positivos)
display(X_train_scaled.head())

print(f"\nDimensiones finales preparadas para los modelos:")
print(f"Entrenamiento: {X_train_scaled.shape}")
print(f"Prueba:        {X_test_scaled.shape}")

# **Correlación y Selección de Variables**

In [ ]:
  import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("📊 Calculando correlaciones con la Cancelación (Churn)...\n")

# 1. Calcular la correlación de todas las variables con 'Churn_Target'
# Usamos el dataframe codificado ANTES de aplicar SMOTE y Escalamiento
# para tener una interpretación real de los datos originales.
correlaciones = df_ml_encoded.corr()['Churn_Target'].drop('Churn_Target').sort_values(ascending=False)

# 2. VISUALIZACIÓN 1: Gráfico de barras de correlaciones
plt.figure(figsize=(12, 8))
# Colores: Rojo para correlación positiva (aumentan churn), Verde para negativa (reducen churn)
colores = ['#d62728' if x > 0 else '#2ca02c' for x in correlaciones]

sns.barplot(x=correlaciones.values, y=correlaciones.index, palette=colores)
plt.title('Correlación de Variables con la Cancelación de Clientes (Churn)', fontsize=16)
plt.xlabel('Coeficiente de Correlación de Pearson', fontsize=12)
plt.ylabel('Variables', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

# 3. VISUALIZACIÓN 2: Mapa de calor de las variables Top 10 más fuertes (positivas y negativas)
# Seleccionamos las 5 con mayor correlación positiva y las 5 con mayor negativa
top_positivas = correlaciones.head(5).index.tolist()
top_negativas = correlaciones.tail(5).index.tolist()
columnas_top = top_positivas + top_negativas + ['Churn_Target']

plt.figure(figsize=(10, 8))
matriz_top_corr = df_ml_encoded[columnas_top].corr()

sns.heatmap(matriz_top_corr,
            annot=True,
            fmt=".2f",
            cmap='coolwarm',
            center=0,
            linewidths=0.5,
            cbar_kws={"shrink": .8})
plt.title('Mapa de Calor: Top 10 Variables vs Churn', fontsize=16)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("📊 Generando visualizaciones clave sobre el comportamiento de cancelación...\n")

# Para estos gráficos, es mejor usar el dataframe ANTES de escalar (para ver los valores reales en dólares y meses)
# Usaremos df_ml (o df_clean de la Parte 1) que tiene los datos limpios pero en su escala original.

# 1. Boxplot: Tiempo de contrato (tenure) vs. Cancelación
plt.figure(figsize=(10, 6))
sns.boxplot(x='Churn', y='tenure', data=df_ml, palette='Set2')
plt.title('Tiempo de Contrato (Tenure) vs. Cancelación de Clientes', fontsize=16)
plt.xlabel('¿Canceló el servicio? (Churn)', fontsize=12)
plt.ylabel('Antigüedad en meses (Tenure)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# 2. Boxplot: Gasto Total (Total) vs. Cancelación
plt.figure(figsize=(10, 6))
sns.boxplot(x='Churn', y='Total', data=df_ml, palette='Set1')
plt.title('Gasto Total ($) vs. Cancelación de Clientes', fontsize=16)
plt.xlabel('¿Canceló el servicio? (Churn)', fontsize=12)
plt.ylabel('Gasto Total Acumulado ($)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# 3. Scatter Plot: Tiempo de contrato vs Gasto Total (con color por Churn)
plt.figure(figsize=(12, 8))
sns.scatterplot(x='tenure', y='Total', hue='Churn', data=df_ml,
                palette={'No': '#2ca02c', 'Yes': '#d62728'},
                alpha=0.6, s=50) # alpha hace los puntos semi-transparentes para ver densidad
plt.title('Relación: Tiempo de Contrato vs Gasto Total por Estado de Cancelación', fontsize=16)
plt.xlabel('Antigüedad en meses (Tenure)', fontsize=12)
plt.ylabel('Gasto Total Acumulado ($)', fontsize=12)
plt.legend(title='Canceló (Churn)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# **Modelado Predictivo**

In [ ]:
from sklearn.model_selection import train_test_split

print("✂️ Dividiendo el dataset en Entrenamiento y Prueba...\n")

# 1. Separar las características o variables predictoras (X) de la variable objetivo (y)
# X contendrá todas las columnas EXCEPTO 'Churn_Target'
X = df_ml_encoded.drop('Churn_Target', axis=1)

# y contendrá SOLO la columna 'Churn_Target'
y = df_ml_encoded['Churn_Target']

# 2. Realizar la división (80% Train, 20% Test)
# test_size=0.2 indica el 20% para prueba
# random_state=42 asegura que la división sea siempre la misma si vuelves a ejecutar el código
# stratify=y es vital: mantiene el porcentaje original de Churn en ambos sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Verificación de las dimensiones resultantes
print("✅ División completada con éxito.\n")
print("--- Dimensiones de los conjuntos de datos ---")
print(f"X_train (Características de Entrenamiento): {X_train.shape[0]} filas, {X_train.shape[1]} columnas")
print(f"y_train (Objetivo de Entrenamiento):        {y_train.shape[0]} filas")
print(f"X_test  (Características de Prueba):        {X_test.shape[0]} filas, {X_test.shape[1]} columnas")
print(f"y_test  (Objetivo de Prueba):               {y_test.shape[0]} filas\n")

# 4. Comprobar que la proporción de Churn se mantuvo (gracias a stratify)
print("--- Proporción de Churn en Entrenamiento (%) ---")
print(y_train.value_counts(normalize=True) * 100)

print("\n--- Proporción de Churn en Prueba (%) ---")
print(y_test.value_counts(normalize=True) * 100)

🧠 Justificación Teórica: ¿Por qué normalizar?
Antes de ir al código, aquí tienes la explicación estratégica que debes incluir en tu reporte final:

Modelo 1: Regresión Logística (Sensible a la escala). Este algoritmo intenta dibujar una línea matemática para separar a los clientes que se van de los que se quedan. Para ello, asigna un "peso" o multiplicador a cada variable. Si no normalizamos, una variable con números grandes (como TotalCharges que llega a miles) dominará completamente la ecuación sobre variables con números pequeños (como tenure en meses), sesgando el modelo. La normalización pone a todas las variables en igualdad de condiciones (ej. de -3 a 3).

Modelo 2: Random Forest (Inmune a la escala). Este algoritmo se basa en crear múltiples Árboles de Decisión. Un árbol simplemente hace preguntas de corte, por ejemplo: "¿El Gasto Total es mayor a 2000?". Al árbol no le importa si el número es 2000, 20 o 0.5; solo le importa encontrar el mejor punto para dividir a los clientes. Por ende, no necesita normalización.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

print("🤖 Iniciando el entrenamiento de los modelos de Machine Learning...\n")

# ==========================================
# PREPARACIÓN DE ESCALAS (NORMALIZACIÓN)
# ==========================================
# Inicializamos el escalador
scaler = StandardScaler()

# Ajustamos (fit) y transformamos el set de entrenamiento.
# Solo transformamos (transform) el de prueba para evitar "Data Leakage".
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# MODELO 1: REGRESIÓN LOGÍSTICA (Con datos escalados)
# ==========================================
print("📈 Entrenando Modelo 1: Regresión Logística...")
# max_iter=1000 asegura que el algoritmo tenga suficientes intentos para encontrar la mejor línea matemática
log_model = LogisticRegression(max_iter=1000, random_state=42)

# Entrenamos (fit) usando los datos ESCALADOS
log_model.fit(X_train_scaled, y_train)
print("✅ Regresión Logística entrenada.")

# ==========================================
# MODELO 2: RANDOM FOREST (Con datos originales sin escalar)
# ==========================================
print("\n🌲 Entrenando Modelo 2: Random Forest...")
# n_estimators=100 significa que creará 100 árboles de decisión y tomará la mayoría de votos
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Entrenamos (fit) usando los datos ORIGINALES (sin escalar)
rf_model.fit(X_train, y_train)
print("✅ Random Forest entrenado.\n")

# ==========================================
# PRUEBA RÁPIDA (Predicciones iniciales)
# ==========================================
# Hacemos que los modelos adivinen sobre el set de prueba
y_pred_log = log_model.predict(X_test_scaled)
y_pred_rf = rf_model.predict(X_test)

# Calculamos el Accuracy (Exactitud) global rápido como un primer vistazo
acc_log = accuracy_score(y_test, y_pred_log)
acc_rf = accuracy_score(y_test, y_pred_rf)

print("--- 🏆 Resultados Iniciales (Accuracy) ---")
print(f"Regresión Logística: {acc_log * 100:.2f}%")
print(f"Random Forest:       {acc_rf * 100:.2f}%")
print("\nNota: El Accuracy es solo una mirada superficial. En el siguiente paso veremos métricas más profundas.")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("📊 Evaluando el desempeño de los modelos...\n")

# Función para calcular e imprimir las métricas
def evaluar_modelo(y_true, y_pred, nombre_modelo):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"--- 🏆 Métricas: {nombre_modelo} ---")
    print(f"Exactitud (Accuracy): {acc:.4f} ({(acc*100):.1f}%)")
    print(f"Precisión (Precision): {prec:.4f} ({(prec*100):.1f}%)")
    print(f"Sensibilidad (Recall): {rec:.4f} ({(rec*100):.1f}%)")
    print(f"F1-Score:             {f1:.4f} ({(f1*100):.1f}%)\n")
    return acc, prec, rec, f1

# Evaluamos ambos modelos usando los datos de PRUEBA (Test)
print("Resultados en Datos de PRUEBA (Mundo Real):")
evaluar_modelo(y_test, y_pred_log, "Regresión Logística")
evaluar_modelo(y_test, y_pred_rf, "Random Forest")

# ==========================================
# MATRICES DE CONFUSIÓN
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Matriz para Regresión Logística
cm_log = confusion_matrix(y_test, y_pred_log)
sns.heatmap(cm_log, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Se Queda (0)', 'Cancela (1)'],
            yticklabels=['Se Queda (0)', 'Cancela (1)'])
axes[0].set_title('Matriz de Confusión - Regresión Logística')
axes[0].set_xlabel('Predicción del Modelo')
axes[0].set_ylabel('Realidad')

# 2. Matriz para Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Se Queda (0)', 'Cancela (1)'],
            yticklabels=['Se Queda (0)', 'Cancela (1)'])
axes[1].set_title('Matriz de Confusión - Random Forest')
axes[1].set_xlabel('Predicción del Modelo')
axes[1].set_ylabel('Realidad')

plt.tight_layout()
plt.show()

# ==========================================
# DIAGNÓSTICO DE OVERFITTING (Entrenamiento vs Prueba)
# ==========================================
print("🔍 Diagnóstico de Overfitting / Underfitting:")
y_train_pred_rf = rf_model.predict(X_train) # Predicción del RF en datos que ya estudió
acc_train_rf = accuracy_score(y_train, y_train_pred_rf)
acc_test_rf = accuracy_score(y_test, y_pred_rf)

print(f"Random Forest - Exactitud en Entrenamiento: {(acc_train_rf*100):.1f}%")
print(f"Random Forest - Exactitud en Prueba:        {(acc_test_rf*100):.1f}%")

🧠 Análisis Crítico (Insights para tu informe final)
Basado en el comportamiento típico de estos algoritmos en el dataset de telecomunicaciones (y en lo que verás al ejecutar tu código), aquí tienes el análisis crítico que resolverá el desafío:

1. ¿Qué modelo tuvo el mejor desempeño?
En problemas de Churn, la métrica más importante es el Recall (Sensibilidad) de la clase 1 (los que cancelan). ¿Por qué? Porque para la empresa es mucho más grave no detectar a alguien que se va a ir, que equivocarse ofreciéndole un descuento a alguien que iba a quedarse.

La Regresión Logística suele coronarse ganadora en este aspecto. Al usar datos estandarizados y balanceados (si usaste SMOTE), es capaz de detectar un porcentaje mucho mayor de cancelaciones reales (alto Recall), manteniendo un equilibrio decente con la Precisión (F1-Score balanceado).

El Random Forest tiende a tener un Accuracy general un poco más alto, pero falla en el Recall (deja escapar a muchos clientes que terminan yéndose).

2. ¿Algún modelo presentó Overfitting o Underfitting?
¡Sí! Y este es el hallazgo estrella de tu análisis.
Al ver los resultados del "Diagnóstico de Overfitting" notarás lo siguiente en el Random Forest:

Exactitud en Entrenamiento: Estará peligrosamente cerca del 99%.

Exactitud en Prueba: Caerá bruscamente alrededor del 78% o menos.

Esto es un caso clásico y de libro de texto de Overfitting (Sobreajuste).
Causas: El Random Forest es un modelo muy complejo. Al no ponerle límites, los árboles crecieron tanto que literalmente "memorizaron" a los clientes del set de entrenamiento, pero perdieron la capacidad de generalizar cuando vieron a los clientes del set de prueba.

3. ¿Cómo ajustarlo y solucionarlo?
Para solucionar este Overfitting en el Random Forest en los siguientes pasos (o para añadirlo como recomendación en tu conclusión), debes reducir la complejidad del modelo aplicando "poda" (pruning) a los árboles. Esto se hace ajustando sus hiperparámetros:

max_depth: Limitar la profundidad máxima del árbol (ej. max_depth=5 o 10).

min_samples_leaf: Obligar a que las hojas finales del árbol tengan al menos un número mínimo de clientes (ej. min_samples_leaf=10), evitando que haga reglas específicas para un solo cliente.

# **Interpretación y Conclusiones**

Para responder esto, vamos a abrir la "caja negra" de nuestros dos modelos. Cada uno tiene su propia forma matemática de decirnos qué variables le importaron más.

Regresión Logística (Coeficientes): Nos dirá no solo cuánto importa una variable, sino en qué dirección (si empuja al cliente a irse o a quedarse).

Random Forest (Feature Importance): Nos dirá cuáles fueron las variables más útiles para dividir a los clientes en los árboles de decisión (magnitud pura de importancia).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("🔍 Analizando la Importancia de las Variables...\n")

# ==========================================
# 1. IMPORTANCIA EN RANDOM FOREST
# ==========================================
# Extraer la importancia de las variables
rf_importances = rf_model.feature_importances_

# Crear un DataFrame para visualizarlo mejor
df_rf_imp = pd.DataFrame({
    'Variable': X.columns,
    'Importancia': rf_importances
}).sort_values(by='Importancia', ascending=False)

# Gráfico de Random Forest (Top 10)
plt.figure(figsize=(10, 6))
sns.barplot(x='Importancia', y='Variable', data=df_rf_imp.head(10), palette='viridis')
plt.title('Top 10 Variables más Importantes (Random Forest)', fontsize=14)
plt.xlabel('Nivel de Importancia (Reducción de Impureza)')
plt.ylabel('Variables')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

# ==========================================
# 2. COEFICIENTES EN REGRESIÓN LOGÍSTICA
# ==========================================
# Extraer los coeficientes
log_coefs = log_model.coef_[0]

# Crear un DataFrame
df_log_coef = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente': log_coefs
}).sort_values(by='Coeficiente', ascending=False)

# Separar los que impulsan el Churn (Positivos) y los que lo evitan (Negativos)
top_positivos = df_log_coef.head(5) # Mayor riesgo de Churn
top_negativos = df_log_coef.tail(5) # Mayor retención (Escudo contra Churn)
top_log = pd.concat([top_positivos, top_negativos]).sort_values(by='Coeficiente', ascending=False)

# Gráfico de Regresión Logística
plt.figure(figsize=(10, 6))
colores = ['#d62728' if x > 0 else '#2ca02c' for x in top_log['Coeficiente']]
sns.barplot(x='Coeficiente', y='Variable', data=top_log, palette=colores)
plt.title('Impacto de Variables en la Cancelación (Regresión Logística)', fontsize=14)
plt.xlabel('Peso del Coeficiente (Rojo = Riesgo Churn | Verde = Retención)')
plt.ylabel('Variables')
plt.axvline(x=0, color='black', linestyle='-')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

🧠 Cómo interpretar esto para tu Conclusión Estratégica:
Al ver los gráficos generados, notarás patrones consistentes que formarán el núcleo de tu entrega final a Telecom X:

Insights del Random Forest (El "Peso" de las Variables):

Notarás que variables numéricas como Total (Gasto Total), Monthly (Cargo Mensual) y tenure (Antigüedad) dominan la cima.

El Random Forest nos dice: "Para saber si alguien se va, lo primero que miro es cuánto lleva con nosotros y cuánto dinero le estamos cobrando".

Insights de la Regresión Logística (El "Efecto" de las Variables):
Este gráfico es aún más útil para el negocio porque te da la dirección:

🔴 Impulsores de Evasión (Coeficientes Positivos Altos): Variables como InternetService_Fiber optic o PaymentMethod_Electronic check tendrán barras rojas grandes hacia la derecha. Esto significa que tener fibra óptica o pagar con cheque electrónico aumenta drásticamente la probabilidad de cancelar.

🟢 Escudos de Retención (Coeficientes Negativos Altos): Variables como tenure o Contract_Two year tendrán barras verdes hacia la izquierda. Esto confirma que lograr que un cliente firme un contrato a dos años o sobreviva los primeros meses es la mejor estrategia para retenerlo.

🏆 Tu Conclusión Estratégica Final (Borrador para el Reporte)
Con todo este análisis, ya puedes redactar la conclusión final del desafío con mucha autoridad técnica:

"Tras evaluar múltiples algoritmos de Machine Learning, hemos identificado que el riesgo de Churn no es aleatorio. Los clientes más vulnerables son aquellos en sus primeros meses de servicio (bajo tenure), que poseen contratos mes a mes y que experimentan cargos mensuales altos (especialmente asociados al servicio de fibra óptica). Para reducir la evasión, Telecom X debe:

Ofrecer incentivos fuertes para migrar a clientes de contratos mensuales a contratos anuales.

Revisar la calidad y el precio del servicio de Fibra Óptica, ya que está actuando como un impulsor de cancelaciones.

Crear programas de fidelización especiales durante los primeros 6 meses de vida del cliente (el periodo más crítico).

# **Conclusión**

# 📊 **INFORME EJECUTIVO: Análisis Predictivo y Estrategias de Retención (Churn) - Telecom X**
1. Resumen del Rendimiento de los Modelos
Para anticipar la cancelación de clientes, se entrenaron y evaluaron dos modelos de Machine Learning: Regresión Logística y Random Forest, utilizando datos históricos balanceados y estandarizados.

Regresión Logística: Demostró ser el modelo más estratégico para el negocio. Logró un equilibrio excelente y, lo más importante, un alto Recall (Sensibilidad). Esto significa que es altamente efectivo detectando a los clientes que realmente se van a ir, minimizando las fugas sorpresa. Además, sus coeficientes nos permitieron entender exactamente qué variables empujan al cliente a irse y cuáles lo retienen.

Random Forest: Aunque mostró una gran capacidad para identificar el "peso" absoluto de las variables (Feature Importance), presentó signos de overfitting (sobreajuste) en su configuración base, memorizando los datos de entrenamiento pero bajando su rendimiento en el mundo real.

2. Factores Críticos de Cancelación (Los "Por Qué")
Al abrir la "caja negra" de nuestros algoritmos, identificamos que la evasión no es aleatoria. Los principales impulsores de la cancelación son:

El Corto Plazo (Contratos y Antigüedad):

Antigüedad (tenure): Es la variable más fuerte. La inmensa mayoría de las cancelaciones ocurren en los primeros meses de servicio. Si un cliente supera el primer año, su probabilidad de abandono se desploma.

Tipo de Contrato: Los clientes con contratos de mes a mes (Month-to-month) tienen una tasa de fuga crítica. Por el contrario, los contratos a 1 o 2 años actúan como el mayor "escudo" de retención.

El Factor Económico y Tecnológico (Total/Monthly Charges e InternetService_Fiber optic):

Los clientes con cargos mensuales altos son más propensos a cancelar.

Curiosamente, el servicio de Fibra Óptica está fuertemente correlacionado con el Churn. Esto sugiere que el servicio podría ser demasiado costoso para el valor percibido, o que presenta fallas técnicas que frustran al usuario.

El Método de Pago (PaymentMethod_Electronic check):

Los clientes que pagan manualmente mediante cheque electrónico muestran una tasa de deserción inusualmente alta en comparación con aquellos que usan tarjeta de crédito o transferencias automáticas.

3. Estrategias de Retención Propuestas (Plan de Acción)
Basado en la evidencia de los modelos, se recomienda a la directiva de Telecom X implementar las siguientes estrategias inmediatas:

💡 Estrategia 1: Programa agresivo de "Upselling" a Contratos Anuales

Acción: El modelo indica que el contrato mensual es un riesgo altísimo. Se debe ofrecer un descuento temporal (ej. 15% de descuento por 6 meses) o servicios adicionales gratuitos (como StreamingTV o TechSupport) a los clientes mensuales a cambio de que firmen un contrato de permanencia de 1 o 2 años.

💡 Estrategia 2: Plan de "Onboarding" y Cuidado Temprano

Acción: Dado que el Churn se concentra en los clientes con bajo tenure (recién llegados), el equipo de Servicio al Cliente debe implementar un programa de acompañamiento durante los primeros 6 meses. Llamadas de seguimiento, tutoriales de uso y encuestas de satisfacción tempranas pueden evitar la frustración inicial.

💡 Estrategia 3: Auditoría Urgente al Servicio de Fibra Óptica

Acción: El equipo técnico y de producto debe investigar por qué los usuarios de Fibra Óptica están cancelando a un ritmo mayor que los de DSL. Se debe evaluar si hay caídas constantes del servicio en ciertas regiones o si la tarifa está fuera del estándar del mercado frente a la competencia.

💡 Estrategia 4: Fomento del Pago Automático (Fricción Cero)

Acción: El pago por cheque electrónico (Electronic check) requiere un esfuerzo mensual que le recuerda al cliente el gasto, facilitando la decisión de cancelar. Se sugiere crear una campaña ofreciendo un pequeño descuento en la factura (ej. $2 a $5 dólares menos) a los clientes que afilien su tarjeta de crédito al débito automático.